In [ ]:
from datasets import load_from_disk
import numpy as np
import pickle

In [ ]:
# Load (wrong) Huggingface dataset
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_old.dataset'
ds = load_from_disk(data_path)
ds

In [2]:
# Load (fixed) Huggingface dataset
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
ds = load_from_disk(data_path)
ds

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 88626282
})

In [ ]:
def _reorder_pair(xs, ys):
    # Stable sort by squared radius without NumPy overhead
    order = sorted(range(len(xs)), key=lambda i: xs[i]*xs[i] + ys[i]*ys[i])
    return [xs[i] for i in order], [ys[i] for i in order]

def reorder_batch(batch):
    rx, ry = batch["rel_x_coord"], batch["rel_y_coord"]
    new_x, new_y = [], []
    append_x, append_y = new_x.append, new_y.append  # tiny micro-opt
    for xs, ys in zip(rx, ry):
        xs_sorted, ys_sorted = _reorder_pair(xs, ys)
        append_x(xs_sorted); append_y(ys_sorted)
    batch["rel_x_coord"] = new_x
    batch["rel_y_coord"] = new_y
    return batch

# Tune these for your machine and IO:
NUM_PROC = 208          # e.g., number of CPU cores
READ_BATCH = 1000     # how many rows processed per function call
WRITE_BATCH = 10000    # how many rows written per Arrow chunk

ds = ds.map(
    reorder_batch,
    batched=True,
    batch_size=READ_BATCH,
    writer_batch_size=WRITE_BATCH,
    num_proc=NUM_PROC,
    desc="Reordering rel_x_coord / rel_y_coord by radius"
)

In [ ]:
item = ds[0]

In [ ]:
item['rel_x_coord']

In [ ]:
item['rel_y_coord']

In [ ]:
data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_90m/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_new.dataset'
ds.save_to_disk(data_path, num_shards=4000)
print("Saved to:", data_path)

In [3]:
# Retrieve indices for reordering
with open('/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_loader_idx.pkl', "rb") as f: 
    indices = pickle.load(f)

ds_all_indices = list(range(len(dataset)))

# Add test and val indices at end of training indices
seen = set(indices)
to_add = []
for idx in ds_all_indices:
    if idx not in seen:
        to_add.append(idx)
        seen.add(idx)

indices.extend(to_add)

In [ ]:
cache_name = "/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_90m_32_None_None_None_None_None_shifted_log_knn_10_loader_reidx.arrow"
ds_ordered = dataset.select(indices,
                            indices_cache_file_name=cache_name)

# Materialize the new physical order
ds_ordered = ds_ordered.flatten_indices(num_proc=208, writer_batch_size=10000)

Flattening the indices (num_proc=208):   5%|▍         | 4296000/88626282 [07:11<6:14:46, 3750.22 examples/s] 

In [ ]:
new_path = data_path.replace('.dataset', '_ordered2.dataset')
ds_ordered.save_to_disk(new_path, num_shards=32)
print("Saved to:", new_path)